In [2]:
import ee
import geemap

ee.Initialize(project='engaged-card-321722')

# Define city boundary (example: Toronto)
city = ee.FeatureCollection("FAO/GAUL/2015/level2") \
         .filter(ee.Filter.eq('ADM2_NAME', 'Toronto'))

Map = geemap.Map()
Map.centerObject(city, 11)
Map.addLayer(city, {}, 'City Boundary')
Map

Map(center=[43.72644545213392, -79.39038941558385], controls=(WidgetControl(options=['position', 'transparent_…

In [3]:
def get_ndvi_composite(start_date, end_date, region):
    """
    Load Sentinel-2 SR imagery and return median NDVI composite.
    Filters clouds using the QA60 band.
    """
    def mask_clouds(image):
        qa = image.select('QA60')
        cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(
                     qa.bitwiseAnd(1 << 11).eq(0))
        return image.updateMask(cloud_mask)

    def add_ndvi(image):
        ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
        return image.addBands(ndvi)

    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                    .filterDate(start_date, end_date)
                    .filterBounds(region)
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                    .map(mask_clouds)
                    .map(add_ndvi))

    return collection.select('NDVI').median().clip(region)

# Get composites for two time periods (summer months for leaf-on)
ndvi_2019 = get_ndvi_composite('2019-06-01', '2019-09-01', city)
ndvi_2023 = get_ndvi_composite('2023-06-01', '2023-09-01', city)

In [4]:
def classify_canopy(ndvi_image, threshold=0.4):
    """
    Binary classification: 1 = canopy, 0 = no canopy
    NDVI > 0.4 is a commonly used threshold for tree canopy
    """
    return ndvi_image.gt(threshold).rename('canopy')

canopy_2019 = classify_canopy(ndvi_2019)
canopy_2023 = classify_canopy(ndvi_2023)

In [6]:
change = canopy_2023.subtract(canopy_2019).rename('change')

vis_change = {
    'min': -1, 'max': 1,
    'palette': ['#d73027', '#f7f7f7', '#1a9850']
}

Map.addLayer(change, vis_change, 'Canopy Change 2019-2023')
Map

Map(bottom=191615.00064087304, center=[43.6663812090663, -79.2241566270578], controls=(WidgetControl(options=[…

In [7]:
def calculate_area_stats(canopy_image, region, scale=10):
    """Calculate canopy area in km²"""
    area_image = canopy_image.multiply(ee.Image.pixelArea())
    stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region.geometry(),
        scale=scale,
        maxPixels=1e10
    )
    area_m2 = stats.getInfo()['canopy']
    return area_m2 / 1e6

area_2019 = calculate_area_stats(canopy_2019, city)
area_2023 = calculate_area_stats(canopy_2023, city)

print(f"Canopy 2019: {area_2019:.2f} km²")
print(f"Canopy 2023: {area_2023:.2f} km²")
print(f"Net change:  {area_2023 - area_2019:.2f} km²")

Canopy 2019: 346.00 km²
Canopy 2023: 340.32 km²
Net change:  -5.68 km²


In [9]:
geemap.ee_export_image_to_drive(
    change,
    description='canopy_change_toronto',
    folder='GEE_exports',
    region=city.geometry(),
    scale=10
)